# 07 - Scene-state and event sidecars (v1.2+)

**Audience:** anyone using the v1.2 scene-recording sidecars.

When scene recording is enabled, each session writes
`<session>_SceneState.csv` (per-frame Recordable transforms) and
`<session>_SceneEvents.csv` (Spawn/Despawn, Config*, ShowInstruction,
CountingAnswerSelected, ChangeDetectionFeedback, RandomStateSnapshot, ...).

## What you'll get

- Sidecar load + metadata read.
- Decoded Config payloads for the four phases.
- Event-type histogram.
- Spawn/Despawn lifetime per ObjectId.
- Gaze x scene-state join: which Recordable was at the gazed-at
  position frame-by-frame?

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Load both sidecars

In [ ]:
if ctx.scene_state_path is None and ctx.scene_events_path is None:
    raise RuntimeError(
        "No sidecars next to this CSV. This notebook requires a v1.2+ "
        "recording. The bundled sample at "
        "Assets/StreamingAssets/EyeTracking_20260503_160712.csv has them."
    )
state, events = ela.load_scene_sidecars(ctx.csv_path, decode_config=True)
print(f"Scene state: {len(state):>6} rows  ({state['ObjectId'].nunique() if not state.empty else 0} unique ObjectIds)")
print(f"Events:      {len(events):>6} rows  ({events['EventType'].nunique() if not events.empty else 0} event types)")
print(f"\nSceneState metadata:")
for k, v in ela.read_csv_metadata(ctx.scene_state_path).items():
    print(f"  {k}: {v}")

## 2. Decoded Config payloads

In [ ]:
if events.empty or "Config" not in events.columns:
    print("No Config events.")
else:
    cfgs = events[events["EventType"].str.startswith("Config", na=False)]
    for _, row in cfgs.iterrows():
        print(f"\n[{row['EventType']}]  T={row['T']:.3f}")
        cfg = row.get("Config")
        if isinstance(cfg, dict):
            for k, v in cfg.items():
                print(f"    {k:<28} {v}")

## 3. Event-type histogram

In [ ]:
if events.empty:
    print("No events.")
else:
    counts = events["EventType"].value_counts()
    fig, ax = plt.subplots(figsize=(10, max(3, 0.25 * len(counts))))
    counts.iloc[::-1].plot(kind="barh", ax=ax)
    ax.set_xlabel("count")
    ax.set_title("Event types")
    plt.tight_layout()
    plt.show()

## 4. Spawn/Despawn lifetimes

In [ ]:
if events.empty:
    print("No events.")
else:
    spawns = events[events["EventType"] == "Spawn"][["T", "ObjectId"]].rename(columns={"T": "spawn_t"})
    despawns = events[events["EventType"] == "Despawn"][["T", "ObjectId"]].rename(columns={"T": "despawn_t"})
    if spawns.empty:
        print("No Spawn events.")
    else:
        joined = spawns.merge(despawns, on="ObjectId", how="left")
        joined["lifetime_s"] = joined["despawn_t"] - joined["spawn_t"]
        live_at_end = joined["despawn_t"].isna().sum()
        finite = joined.dropna(subset=["lifetime_s"])
        print(f"Spawn events:        {len(joined)}")
        print(f"Still alive at end:  {live_at_end}")
        if not finite.empty:
            print(f"Lifetime - median {finite['lifetime_s'].median():.2f}s  p95 {finite['lifetime_s'].quantile(0.95):.2f}s  max {finite['lifetime_s'].max():.2f}s")

## 5. Gaze x scene-state join

Each gaze frame gets the poses of every Recordable present that frame.
Result is many-to-many on Frame (so larger than the gaze DataFrame).

In [ ]:
if state.empty:
    print("No scene state.")
else:
    df = ctx.data.df
    if "frame_number" not in df.columns:
        print("Gaze CSV missing frame_number column.")
    else:
        joined = ela.merge_gaze_with_scene_state(df, state)
        print(f"Gaze rows:        {len(df)}")
        print(f"Joined rows:      {len(joined)}")
        print(f"Avg recordables/frame: {len(joined)/max(1, len(df)):.2f}")
        # Most-frequently-present recordable
        if "ObjectId" in joined.columns:
            top = joined["ObjectId"].value_counts().head(5)
            print("\nTop 5 recordables by frame count:")
            for oid, n in top.items():
                if pd.isna(oid):
                    continue
                print(f"  {oid}: {n} frames")